# 03 Extraction: roles, skills and requirements

This notebook turns the data-role ads from `02_cleaning` into structured information for the analysis (`04_analysis`) and the web app:

1. **Role and seniority** from the job title
2. **Skills** from the full ad text
3. **Required vs. nice-to-have** skills, from hedging words around each mention
4. **Entry barriers:** experience, degree and German level

Each extraction step is checked by hand on a random sample, and the judgments are recorded in the notebook.

**Input:** `data/processed/jobs_data_roles.pkl` (1,068 ads)  
**Output:** `data/processed/jobs_extracted.pkl` (1,062 ads, one row per ad), `reports/figures/top_skills.png`

**Contents:** 1 Setup · 2 Cleaning the ad texts · 3 Ads matched only through a company name · 4 Role · 5 Seniority · 6 Skills · 7 Required vs. nice-to-have · 8 Entry barriers · 9 Checkpoint · 10 Summary and future work

## 1. Setup

In [ ]:
from pathlib import Path
import html
import re

import matplotlib.pyplot as plt
import pandas as pd

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
FIG_DIR = Path.cwd().parent / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

jobs = pd.read_pickle(PROCESSED_DIR / "jobs_data_roles.pkl")
print(jobs.shape)
print(jobs.columns.tolist())

## 2. Cleaning the ad texts

`clean_text` decodes HTML entities and removes Markdown markers, but keeps line breaks and bullet points: the structure of the ads is needed later to tell tasks from requirements.

In [ ]:
def clean_text(s):
    """Readable ad text: decode HTML entities, remove Markdown markers, keep line breaks."""
    return (s.map(html.unescape)
             .str.replace(r"\\([*_])", r"\1", regex=True)               # unescape \* and \_
             .str.replace(r"\*\*|__", "", regex=True)                   # bold markers
             .str.replace(r"(?m)^[ \t]*#+[ \t]*", "", regex=True)       # heading markers
             .str.replace(r"(?m)^[ \t]*[*•][ \t]+", "- ", regex=True)   # bullets -> "- "
             .str.replace(r"[ \t]+", " ", regex=True)                   # collapse spaces
             .str.replace(r"\n\s*\n+", "\n", regex=True)                # drop empty lines
             .str.strip())

jobs["text"] = clean_text(jobs["stellenangebotsBeschreibung"])

print(jobs["text"].str.len().describe().round())
print("\n--- example ---\n")
print(jobs["text"].iloc[0][:800])

**The cleaned texts are readable and keep their structure.** The example already shows the two things the extraction relies on:

- **Section headings** separate tasks ("Ihre Aufgaben bei uns") from requirements ("Was wir von Ihnen erwarten").
- **Hedging words mark optional requirements:** "Berufserfahrung in SAP, *idealerweise* mit SAP Business One".

Skills appear in both sections: here, Power BI, SQL, Power Query and DAX are already named in the tasks. So skills are searched in the whole text, not only in the requirements.

**The ads are long:** the median text has 3,569 characters, and the longest 9,908. The shortest has only 247 characters, so a few ads may contain too little text to extract much from.

## 3. Ads that matched only through a company name

The employer NTT DATA puts its name in front of every job title ("NTT DATA Deutschland SE: …"), and the word "DATA" in that prefix was enough to pass the data-role filter in `02_cleaning`. The cell below strips the prefix and checks the rest of the title with the same data terms as the final filter. The title without the prefix (`title`) is also used for roles and seniority.

In [ ]:
DATA_TITLE_V3 = (r"data|daten(?!schutz)|analy|statisti|intelligen"
                 r"|\bbi\b|\bsap bw|datasphere|snowflake"
                 r"|machine.?learning|maschinell|deep.?learning|computer.?vision"
                 r"|\bml|mlops|\bki\b|\bai\b|genai|\bllm|\bnlp\b|prompt.?engineer|\betl\b"
                 r"|applied.?scien|decision.?scien")   # same data terms as the final rule in 02_cleaning

# Job title without the company prefix "NTT DATA …:", used for all title rules in this notebook
title = jobs["stellenangebotsTitel"].str.replace(r"^NTT DATA[^:]*:\s*", "", case=False, regex=True)

ntt = jobs["stellenangebotsTitel"].str.contains("NTT DATA", case=False)
company_only = ntt & ~title.str.contains(DATA_TITLE_V3, case=False, regex=True)

print(ntt.sum(), "NTT DATA ads,", company_only.sum(), "without a data term outside the company name:")
print(jobs.loc[company_only, "stellenangebotsTitel"].to_string(), "\n")

jobs = jobs[~company_only].reset_index(drop=True)
title = title[~company_only].reset_index(drop=True)
print(jobs.shape)

**Six ads were in the dataset only because the employer is called "NTT DATA".** Their titles contain no data term, e.g. "Client Partner Banking DACH", "Consultant DevSecOps/Application Security" or "Linux Administrator:in im Storage". Removing them leaves 11 of the 17 NTT DATA ads and 1,062 ads in total.

*Future work:* the prefix removal belongs in the filter in `02_cleaning`, so that the cleaning notebook itself is correct.

## 4. Role from the title

**Each ad gets exactly one role.** The rules are checked in a fixed order and the first match wins: named roles first (Data Scientist, Data Engineer, ML / AI Engineer, BI / Analytics, Data Analyst), then two generic rules. A title with a data word *and* an engineering word ("Data Software Engineer", "Data Architekt") counts as Data Engineer; a title with only a generic AI term ("AI Engineer", "Künstliche Intelligenz") counts as ML / AI Engineer. Titles that match no rule become *Other data role*.

There are no hand labels for the roles, so the cell also runs four checks on the result.

In [ ]:
# First match wins: named roles first, generic rules only at the end
ROLES = {
    "Data Scientist":   r"data.?scien|datenwissenschaft|applied.?scien|decision.?scien",
    "Data Engineer":    (r"data.?engineer|dateningenieur|analytics.?engineer|\betl\b|data.?platform|snowflake"
                         r"|data.?integration|data.?architect|dataarchitekt|datenarchitekt|databricks|data.?pipeline"),
    "ML / AI Engineer": r"machine.?learning|maschinell|\bml\b|mlops|deep.?learning|computer.?vision|\bllm|\bnlp\b",
    "BI / Analytics":   (r"\bbi\b|business.?intelligen|power.?bi|\bsap bw|data.?warehouse|datasphere"
                         r"|sap analytics|\bsac\b|tableau|qlik|microstrategy|looker"),
    "Data Analyst":     r"analy|statisti|data.?mining",
    # (?=...) is a lookahead: two together mean "contains a data word AND an engineering word, in any order"
    "Data Engineer (generic)":       r"^(?=.*(?:\bdata|\bdaten))(?=.*(?:engineer|archite[ck]t|entwickler|developer))",
    "ML / AI Engineer (generic AI)": r"\bai\b|\bki\b|genai|prompt.?engineer|künstliche.?intelligenz|artificial.?intelligence",
}

matches = pd.DataFrame({role: title.str.contains(p, case=False, regex=True) for role, p in ROLES.items()})
named = ["Data Scientist", "Data Engineer", "ML / AI Engineer", "BI / Analytics", "Data Analyst"]

# idxmax returns the first matching column, i.e. the first rule in the order above
jobs["role"] = (matches.idxmax(axis=1)
                .where(matches.any(axis=1), "Other data role")
                .replace({"ML / AI Engineer (generic AI)": "ML / AI Engineer",
                          "Data Engineer (generic)": "Data Engineer"}))

print(jobs["role"].value_counts(), "\n")

# Check 1: titles matching more than one role (only the order decides)
n_matches = matches[named + ["ML / AI Engineer (generic AI)"]].sum(axis=1)
print("roles matched per title:")
print(n_matches.value_counts().sort_index())
print((n_matches > 1).sum(), "titles match more than one role\n")

multi = jobs.loc[n_matches > 1, ["stellenangebotsTitel", "role"]].copy()
multi["also matches"] = matches[n_matches > 1].apply(lambda r: ", ".join(r.index[r]), axis=1)
print(multi.sample(min(20, len(multi)), random_state=1).to_string(), "\n")

# Check 2: role vs. official occupation for the most frequent occupations
top_occ = jobs["hauptberuf"].value_counts().head(8).index
sub = jobs[jobs["hauptberuf"].isin(top_occ)]
print(pd.crosstab(sub["hauptberuf"], sub["role"]), "\n")

# Check 3: what is left in "Other" for ads whose occupation is Data Engineer or Data Scientist
left = jobs[(jobs["role"] == "Other data role") & jobs["hauptberuf"].isin(["Data Engineer", "Data Scientist"])]
print(len(left), "ads in 'Other' with occupation Data Engineer / Data Scientist:")
print(left["stellenangebotsTitel"].to_string(), "\n")

# Check 4: ads decided by the generic data-engineering rule
by_generic = matches["Data Engineer (generic)"] & ~matches[named].any(axis=1)
print(by_generic.sum(), "ads assigned by the generic data-engineering rule")
print((by_generic & matches["ML / AI Engineer (generic AI)"]).sum(), "of them also contain a generic AI term\n")
print(jobs.loc[by_generic, "stellenangebotsTitel"].to_string())

| Role | Ads |
|---|---|
| Data Engineer | 281 |
| Data Analyst | 247 |
| ML / AI Engineer | 211 |
| BI / Analytics | 119 |
| Data Scientist | 119 |
| Other data role | 85 |

**The rules were revised three times, each time based on a check of the output:**

- **Generic AI terms came too early:** "Data Analyst & AI Expert" was assigned to ML / AI. They now come last.
- **Data warehousing pulled BI jobs into Data Engineer:** "BI Developer / Data Warehouse Developer". Data warehouse and SAP Datasphere now count as BI.
- **Free-form data engineering titles fell into *Other*:** "Data Migration Engineer", "Data Architekt". The generic data-engineering rule catches 41 of them.

**The checks support the result:**

- **Order matters for few ads:** 773 titles match exactly one role, 172 match two or three, and 117 match none (→ *Other*).
- **The official occupation broadly agrees:** 110 of the ads with the occupation *Data Engineer* are assigned *Data Engineer*, and 62 of the *Data-Analyst/in* ads *Data Analyst*. The occupation field is unreliable for single ads (see `02_cleaning`), so this is a plausibility check, not a validation.
- **The remaining *Other* roles are data management, governance, quality and strategy,** e.g. "Datenmanager", "Data Governance Manager", "Principal Data Strategist", which is what the category is for.

There are no hand labels for the roles, so their accuracy is not measured.

## 5. Seniority from the title

Senior and junior terms are searched in the title. A title with both counts as "junior and senior"; a title with neither as "not stated".

In [ ]:
SENIOR = r"\bsenior\b|\bsr\.?\b|\blead\b|principal|\bstaff\b|head of|leiter"
JUNIOR = r"\bjunior\b|\bjr\.?\b|einsteiger|trainee|graduate|absolvent|entry"

is_senior = title.str.contains(SENIOR, case=False, regex=True)
is_junior = title.str.contains(JUNIOR, case=False, regex=True)

jobs["seniority"] = "not stated"
jobs.loc[is_senior & ~is_junior, "seniority"] = "senior"
jobs.loc[is_junior & ~is_senior, "seniority"] = "junior"
jobs.loc[is_junior & is_senior, "seniority"] = "junior and senior"   # e.g. "(Junior/Senior) Data Analyst"

print(jobs["seniority"].value_counts(), "\n")
print(pd.crosstab(jobs["role"], jobs["seniority"], margins=True), "\n")

# Check 1: levels in parentheses, e.g. "(Senior) Data Scientist" = also open to non-seniors
in_parens = title.str.contains(r"\(\s*(?:senior|sr\.?|junior|jr\.?)\s*\)", case=False, regex=True)
print(in_parens.sum(), "titles with the level in parentheses:")
print(jobs.loc[in_parens, "seniority"].value_counts(), "\n")

# Check 2: 10 sample titles per seniority level
for level in jobs["seniority"].value_counts().index:
    print(f"--- {level}")
    sample = jobs.loc[jobs["seniority"] == level, "stellenangebotsTitel"]
    print(sample.sample(min(10, len(sample)), random_state=1).to_string(), "\n")

**Most titles do not state a level at all: 757 of 1,062 ads.** 250 titles are marked as senior (senior, lead, principal, staff, head of, Leiter) and only 55 as junior (junior, trainee, graduate, Absolvent, Einsteiger, which also matches "Quereinsteiger").

| Role | Junior | Not stated | Senior |
|---|---|---|---|
| BI / Analytics | 5 | 97 | 17 |
| Data Analyst | 25 | 180 | 42 |
| Data Engineer | 7 | 181 | 93 |
| Data Scientist | 7 | 86 | 26 |
| ML / AI Engineer | 8 | 149 | 54 |
| Other data role | 3 | 64 | 18 |

Data Engineer has the most senior titles (93), Data Analyst the most junior titles (25).

**Two limitations:**

- **64 titles give the level in parentheses**, e.g. "(Senior) Data Scientist" (44 senior, 20 junior). This usually means "also open to this level", so these ads are counted as senior or junior although they may accept others.
- **"Expert" is not used as a senior term,** because it also matched company names at the end of titles.

Because so few titles state a level, the years of experience required in the ad text will be the more informative measure of how open ads are to newcomers.

## 6. Skills

### 6.1 Skill dictionary

**Skills are found with a dictionary of 46 skills and their German and English variants.** Each skill is a regular expression searched in the whole ad text, because skills appear in the task section as well as in the requirements.

Three design choices handle the difficult cases:

- **German compounds:** most patterns have no word boundary at the end, so "Python" also matches "Pythonkenntnisse" and "SQL" matches "SQL-Abfragen".
- **Short or ambiguous names** get extra conditions: "R" is matched case-sensitively as a separate word, and not in "R&D", "R+V" or "i. d. R."; "Excel" not in "Excellence"; "Spark" not in "Sparkasse"; "Java" not in "JavaScript"; "DAX" not as the stock index ("DAX-Konzern"); "Agile" only as a method ("agile Methoden"), not as a description of the employer.
- **SQL dialects count as SQL** (MySQL, PostgreSQL), but NoSQL does not.

In [ ]:
# Skill dictionary. Regex symbols used below:
#   (?i)    case-insensitive (only R and SAS are case-sensitive on purpose)
#   \b      word boundary: start or end of a word (space, hyphen, punctuation)
#           -> no \b at the end lets compounds match: "python" also finds "Pythonkenntnisse"
#   .?      one optional character of any kind: "power.?bi" = "Power BI", "Power-BI", "PowerBI"
#   a|b     a or b
#   (?:…)   group alternatives without capturing them
#   (?!…)   negative lookahead: NOT followed by …
#   (?<!…)  negative lookbehind: NOT preceded by …
#   \w*     any number of letters;  \s+ one or more spaces;  \d a digit
#   \+      a literal "+" (a plain + would mean "one or more")
SKILLS = {
    # programming languages
    "Python":                  r"(?i)python",                     # anywhere, incl. "Pythonkenntnisse"
    "SQL":                     r"(?i)(?<!no)sql",                 # SQL, MySQL, PostgreSQL, "SQL-Abfragen"; not NoSQL
    "R":                       r"(?<!d\.)(?<!d\. )\bR\b(?!\s*[&+])",   # not "R&D", "R+V", "i.d.R.", "i. d. R."
    "Java":                    r"(?i)\bjava(?!script)",           # word starting with "java", not JavaScript
    "JavaScript / TypeScript": r"(?i)javascript|typescript",      # either language
    "Scala":                   r"(?i)\bscala\b",                  # whole word only; not "scalable"
    "C++":                     r"(?i)c\+\+",                      # "C++" (the + signs escaped)
    "C#":                      r"(?i)\bc#",                       # "C#" at the start of a word
    "SAS":                     r"\bSAS\b",                        # capital SAS as a separate word
    "MATLAB":                  r"(?i)matlab",                     # anywhere
    "VBA":                     r"(?i)\bvba\b",                    # whole word
    # data engineering
    "Spark":                   r"(?i)\b(?:py)?spark\b",           # "Spark" or "PySpark" as a whole word; not "Sparkasse"
    "Kafka":                   r"(?i)kafka",                      # anywhere
    "Airflow":                 r"(?i)airflow",                    # anywhere
    "dbt":                     r"(?i)\bdbt\b",                    # whole word (the tool "dbt")
    "ETL / ELT":               r"(?i)\betl\b|\belt\b",            # the abbreviations as whole words
    "Data warehouse":          r"(?i)data.?warehous|\bdwh\b",     # "Data Warehouse", "Data-Warehousing", or "DWH"
    "Data modelling":          r"(?i)datenmodell|data.?modell?ing",  # "Datenmodellierung", "data modeling/modelling"
    "Snowflake":               r"(?i)snowflake",                  # anywhere
    "Databricks":              r"(?i)databricks",                 # anywhere
    "NoSQL":                   r"(?i)nosql|mongodb|cassandra",    # NoSQL or two common NoSQL databases
    # cloud and DevOps
    "AWS":                     r"(?i)\baws\b|amazon web services",  # abbreviation or full name
    "Azure":                   r"(?i)\bazure",                    # word starting with "Azure", e.g. "Azure-Cloud"
    "Google Cloud":            r"(?i)\bgcp\b|google cloud",       # abbreviation or full name
    "Docker":                  r"(?i)docker",                     # anywhere
    "Kubernetes":              r"(?i)kubernetes|\bk8s\b",         # full name or the short form "k8s"
    "Terraform":               r"(?i)terraform",                  # anywhere
    "Git":                     r"(?i)\bgit\b|github|gitlab",      # "Git" as a whole word, or GitHub/GitLab; not "digital"
    "Linux":                   r"(?i)linux",                      # anywhere
    # BI and reporting
    "Power BI":                r"(?i)power.?bi",                  # "Power BI", "Power-BI", "PowerBI"
    "Tableau":                 r"(?i)tableau",                    # anywhere
    "Qlik":                    r"(?i)qlik",                       # anywhere, incl. "QlikView", "Qlik Sense"
    "Excel":                   r"(?i)\bexcel(?!len)",             # "Excel", "Excelkenntnisse"; not "Excellence", "excellent"
    "DAX / Power Query":       r"(?i)power.?query|\bdax\b(?![- ]?(?:konzern|unternehmen|notiert|\d))",
                                                                  # Power Query, or "DAX" but not the stock index
                                                                  # ("DAX-Konzern", "DAX-Unternehmen", "DAX 40")
    "SAP":                     r"(?i)\bsap\b",                    # whole word, incl. "SAP-Kenntnisse"
    # machine learning and AI
    "Machine learning":        r"(?i)machine.?learning|maschinelle\w*\s+lernen|\bml\b",
                                                                  # "machine learning", "maschinelles Lernen", or "ML"
    "Deep learning":           r"(?i)deep.?learning|neuronale|neural net",  # incl. "neuronale Netze", "neural networks"
    "NLP":                     r"(?i)\bnlp\b|natural language|sprachverarbeitung",  # abbreviation, English or German term
    "Computer vision":         r"(?i)computer.?vision|bildverarbeitung",           # English or German term
    "LLM / GenAI":             r"(?i)\bllms?\b|large language model|generative (?:ai|ki)|\bgenai\b|\bgpt",
                                                                  # LLM(s), "generative AI/KI", GenAI, words starting with GPT
    "scikit-learn":            r"(?i)scikit|sklearn",             # full name or import name
    "TensorFlow":              r"(?i)tensorflow",                 # anywhere
    "PyTorch":                 r"(?i)pytorch",                    # anywhere
    "pandas":                  r"(?i)\bpandas\b",                 # whole word
    "Statistics":              r"(?i)statisti",                   # "Statistik", "statistical", "Statistiker"
    # ways of working
    "Agile / Scrum":           (r"(?i)\bscrum|\bkanban"
                                r"|\bagil\w*\s+(?:method|projekt|vorgehen|entwicklung|development|framework|practice|praktik)"),
                                                                  # Scrum, Kanban, or "agile Methoden/Projekte/Entwicklung"
}

### 6.2 Extraction

Each skill becomes one True/False column in `S` (one row per ad). `jobs["skills"]` holds the list of skills found in each ad.

In [ ]:
text = jobs["text"]
S = pd.DataFrame({skill: text.str.contains(p, regex=True) for skill, p in SKILLS.items()})
jobs["skills"] = S.apply(lambda row: list(row.index[row]), axis=1)

counts = S.sum().sort_values(ascending=False)   # number of ads that mention each skill
print(len(SKILLS), "skills in the dictionary\n")
print(pd.DataFrame({"ads": counts, "share": (counts / len(jobs)).round(3)}).to_string(), "\n")

per_ad = S.sum(axis=1)                          # number of skills found in each ad
print("skills found per ad:")
print(per_ad.describe().round(1))
print((per_ad == 0).sum(), "ads with no skill found")

### 6.3 Checking the matches

**I read five random hits per skill in their context and counted how many really mean the skill.** A hit counts as correct when the word refers to a tool or method the job uses or requires (in the tasks, the requirements or the team's tech stack), including fields of study such as "Studium der Statistik". It counts as wrong when the word means something else: a company name, an application instruction, or a description of the employer.

The check covers the 15 most frequent skills and all skills with risky patterns (short names such as R, SAS or Git, and words with other meanings). Each match is shown between `>>> <<<`.

In [ ]:
def context(t, pattern, width=60):
    """Return the first match of a pattern with some text around it.

    Args:
        t (str): the text to search in
        pattern (str): the regex pattern to search for
        width (int): number of characters to include on each side of the match

    Returns:
        str: the match, marked with >>> <<<, and its surrounding text on one line
    """
    m = re.search(pattern, t)
    if m is None:   # cannot happen for texts selected by S, but keeps the function safe on its own
        return "…no match found…"
    start = max(0, m.start() - width)   # max() stops the slice at the start of the text
    return ("…" + t[start:m.start()] + " >>>" + m.group() + "<<< " + t[m.end():m.end() + width] + "…").replace("\n", " ")


RISKY = ["R", "SAS", "Excel", "Spark", "DAX / Power Query", "Git", "Scala", "C#",
         "Machine learning", "SAP", "Statistics", "Agile / Scrum", "ETL / ELT"]
to_check = list(dict.fromkeys(list(counts.index[:15]) + RISKY))   # dict.fromkeys removes duplicates, keeps the order

for skill in to_check:
    hits = jobs.loc[S[skill], "text"]
    print(f"=== {skill} ({len(hits)} ads)")
    for t in hits.sample(min(5, len(hits)), random_state=1):   # random_state: the same 5 hits on every run
        print("   ", context(t, SKILLS[skill]))
    print()

In [ ]:
# Manual spot check: correct hits out of 5 per skill, read in the output above (final patterns)
spot_check = {
    "Python": 5, "SQL": 5, "Machine learning": 5, "Data modelling": 5,
    "Power BI": 5, "Azure": 5, "ETL / ELT": 5, "Agile / Scrum": 5,
    "Data warehouse": 5, "SAP": 5, "Statistics": 5, "LLM / GenAI": 5,
    "AWS": 5, "Excel": 5, "Git": 3, "Databricks": 5, "R": 5, "SAS": 5,
    "Spark": 5, "DAX / Power Query": 5, "Scala": 5, "C#": 5,
}

# Results of the earlier pattern versions (version 1, version 2)
spot_check_before = {"Agile / Scrum": [1, 2], "R": [4, 4]}

check = pd.DataFrame({"correct of 5": spot_check}).sort_values("correct of 5")
print(check.to_string())
print("\n", check["correct of 5"].sum(), "of", 5 * len(check), "checked hits correct")
print(len(check), "skills checked,", len(SKILLS) - len(check), "not checked\n")

print("earlier pattern versions (version 1, version 2):")
for skill, results in spot_check_before.items():
    print(f"  {skill}: {results} -> final {spot_check[skill]}")

**108 of 110 checked hits are correct, across 22 skills.** The only skill below 5 of 5 is Git (3 of 5), where two hits are GitHub as a place for application documents. The Git count (114 ads) is therefore likely too high.

**Two patterns were revised during the check:**

| Skill | Version 1 | Version 2 | Final | What changed |
|---|---|---|---|---|
| Agile / Scrum | 1 of 5 | 2 of 5 | 5 of 5 | first any "agil…" word, then "agil…" + a work word (which still matched "agile Arbeitskultur"); now only methods: Scrum, Kanban, "agile Methoden/Projekte" |
| R | 4 of 5 | 4 of 5 | 5 of 5 | "R+V" (an insurer), then "i. d. R." (*in der Regel*), are now excluded |

The 24 skills with unambiguous names (e.g. TensorFlow, Snowflake, Kafka) were not checked.

### 6.4 First result: the most frequent skills

The chart shows the 20 most frequent skills. It is saved to `reports/figures/` for the presentation.

In [ ]:
# Muted, corporate style: desaturated slate blue, white background, soft dark-grey text
SURFACE, INK, INK_2, BAR = "#ffffff", "#2b2b2b", "#6b6b6b", "#5b7a99"
plt.rcParams.update({"font.family": "sans-serif",
                     "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"]})

top = counts.head(20)[::-1]                    # reversed so the most frequent skill is at the top
share = top / len(jobs) * 100
first, second = counts.index[:2]               # title is built from the data, not typed by hand

fig, ax = plt.subplots(figsize=(9, 8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

colors = [BAR if skill in (first, second) else "#c9ced6" for skill in top.index]
ax.barh(top.index, share, height=0.62, color=colors)

# Value at the tip of each bar (replaces the x-axis, so the axis is hidden)
for i, v in enumerate(share):
    ax.text(v + 0.6, i, f"{v:.0f}%", va="center", fontsize=10, color=INK_2)

# Title states the finding; subtitle says exactly what is measured
fig.text(0.02, 0.965, f"{first} and {second} are the most requested skills in German data jobs",
         fontsize=15, fontweight="bold", color=INK, ha="left")
fig.text(0.02, 0.93, f"Share of {len(jobs):,} data-role ads that mention each skill (top 20 of {len(SKILLS)})",
         fontsize=11, color=INK_2, ha="left")

# Source and method note
fig.text(0.02, 0.015,
         "Source: job search of the Bundesagentur für Arbeit, collected 7 October 2026. "
         "A skill counts when it is mentioned anywhere in the ad, including as nice to have.",
         fontsize=8.5, color=INK_2, ha="left")

# Recessive frame: no spines, no x-axis, no ticks
for side in ["top", "right", "bottom", "left"]:
    ax.spines[side].set_visible(False)
ax.xaxis.set_visible(False)
ax.tick_params(axis="y", length=0, labelsize=10.5, labelcolor=INK)
ax.set_xlim(0, share.max() * 1.12)             # room for the value labels

plt.subplots_adjust(left=0.2, right=0.97, top=0.9, bottom=0.06)
fig.savefig(FIG_DIR / "top_skills.png", dpi=200, facecolor=SURFACE)
plt.show()

**Python and SQL stand out: they are mentioned in 46% and 44% of the 1,062 data-role ads.** No other skill reaches a third of the ads. The next group consists of machine learning (30%), data modelling (30%), Power BI, Azure and ETL / ELT (each about 20%).

Two points stand out for the German market:

- **SAP appears in 18% of the ads.** A large part of data work in Germany happens inside SAP environments, which is easy to overlook when looking at international, English-language studies.
- **LLM / GenAI already appears in 14% of the ads**, about as often as AWS (13%).

The chart counts *mentions*: a skill counts as soon as it appears anywhere in the ad, also as "nice to have". The split into required and nice-to-have skills follows in section 7.

## 7. Required vs. nice-to-have

**A skill counts as nice-to-have when the ad hedges it,** e.g. "*idealerweise* Kenntnisse in Spark" or "Experience with dbt *is a plus*". Each ad is split into short units (lines, bullet points, sentences), and a unit counts as hedged when it contains a hedging word. A skill is *nice-to-have* in an ad if all its mentions are in hedged units, and *required* if at least one mention is not. Mentions in the task section count as required, because the job needs the skill.

### 7.1 Splitting the ads into units

Units are lines, bullet points, items separated by ";" and sentences. A unit counts as hedged when it contains a German or English hedging word.

In [ ]:
# Units: lines, ";", inline bullets (" - ", " · ", " • ") and sentence ends.
# A sentence end needs 4 lower-case letters before the full stop, so "z. B." or "bzw." does not split a unit.
SPLIT = r"\n|;|\s[-–•·]\s|(?<=[a-zäöüß]{4}[.!?])\s+(?=[A-ZÄÖÜ])"

HEDGE = (r"(?i)idealerweise|wünschenswert|von vorteil|vorteilhaft|\bein plus\b|pluspunkt|gern gesehen"
         r"|wäre schön|nicht zwingend|kein muss|bevorzugt|vorzugsweise|optional"
         r"|nice.?to.?have|\bideally\b|preferred|preferabl|desirable|advantageous|beneficial|\ba plus\b|\bbonus\b")

units = (jobs["text"].str.split(SPLIT, regex=True)
         .explode()
         .rename("unit")
         .reset_index()
         .rename(columns={"index": "ad"}))
units = units[units["unit"].str.strip().str.len() > 0].reset_index(drop=True)
units["hedged"] = units["unit"].str.contains(HEDGE, regex=True)

print(len(units), "units in", units["ad"].nunique(), "ads")
print(units["hedged"].sum(), "hedged units\n")
print("examples of hedged units:")
print(units.loc[units["hedged"], "unit"].sample(10, random_state=1).to_string())

**The 1,062 ads are split into 42,563 units, and 1,218 of them contain a hedging word.** The examples show the typical cases: "*idealerweise* mit Fokus auf Machine Learning", "*Idealerweise* Erfahrung in der Einführung neuer …".

They also show two limitations of searching for hedging words unit by unit:

- **A hedging word can belong to something other than a skill:** "Bewerbende mit Schwerbehinderung werden … *bevorzugt*" (equal-opportunity statement) or "Du kennst den Aktienmarkt *idealerweise* nicht …". These units contain no skill, so they do not affect the result.
- **Some ads hedge a whole section with a heading,** e.g. "Das i-Tüpfelchen (Nice-to-Have):". The bullet points below such a heading contain no hedging word themselves, so the skills in them are wrongly counted as required.

### 7.2 Hedged sections

To fix the second limitation, a heading that hedges (e.g. "Von Vorteil:") marks every unit below it as hedged, up to the next heading. A heading is a short unit (at most 80 characters) that ends with ":".

In [ ]:
# Hedged sections: a short unit ending in ":" with an explicit nice-to-have word hedges all units below it,
# up to the next heading. "idealerweise" is left out on purpose: "Das bringst du idealerweise mit:"
# often introduces the whole requirements section.
HEDGE_HEADING = (r"(?i)nice.?to.?have|von vorteil|wünschenswert|\bplus\b|pluspunkt|tüpfelchen|\bbonus\b"
                 r"|optional|wäre schön|gern gesehen|desirable|preferred|advantageous|beneficial")

stripped = units["unit"].str.strip()
units["heading"] = (stripped.str.len() <= 80) & stripped.str.endswith(":")
units["section"] = units.groupby("ad")["heading"].cumsum()   # section number within each ad

hedge_headings = units["heading"] & stripped.str.contains(HEDGE_HEADING, regex=True)
hedged_sections = pd.MultiIndex.from_frame(units.loc[hedge_headings, ["ad", "section"]])
units["in_hedged_section"] = (pd.MultiIndex.from_frame(units[["ad", "section"]]).isin(hedged_sections)
                              & ~units["heading"])

print(units["heading"].sum(), "headings,", hedge_headings.sum(), "of them hedge a section")
print(units["in_hedged_section"].sum(), "units below a hedging heading\n")
print("hedging headings:")
print(stripped[hedge_headings].value_counts().head(20).to_string())

units["hedged"] = units["hedged"] | units["in_hedged_section"]

**30 of the 1,457 headings hedge a whole section,** e.g. "Von Vorteil:", "Preferred qualifications:", "Plus, aber kein Muss:" or "Das i-Tüpfelchen (Nice-to-Have):". The 426 units below them now count as hedged as well. "Idealerweise" is not used for headings, because "Das bringst du idealerweise mit:" often introduces the whole requirements section.

The effect is small but goes in the expected direction: e.g. nice-to-have mentions of Python rise from 79 to 83, and of Azure from 36 to 40.

### 7.3 Classifying each skill mention

Each skill pattern is searched unit by unit. A skill is *required* in an ad if at least one mention is in an unhedged unit, and *nice-to-have* if all mentions are hedged. The cell also counts mentions that are found in the full text but in no unit (a pattern split across a unit boundary).

In [ ]:
REQ = pd.DataFrame(False, index=jobs.index, columns=list(SKILLS))
NICE = pd.DataFrame(False, index=jobs.index, columns=list(SKILLS))

for skill, p in SKILLS.items():
    hit = units["unit"].str.contains(p, regex=True)
    req_ads = units.loc[hit & ~units["hedged"], "ad"].unique()
    any_ads = units.loc[hit, "ad"].unique()
    REQ.loc[req_ads, skill] = True
    NICE.loc[any_ads, skill] = True
NICE = NICE & ~REQ   # nice-to-have only if never mentioned unhedged

# Mentions found in the full text but not in any unit (a pattern split across a unit boundary)
lost = S & ~(REQ | NICE)
print(int(lost.values.sum()), "skill mentions not found at unit level\n")

summary = pd.DataFrame({"mentioned": S.sum(), "required": REQ.sum(), "nice-to-have": NICE.sum()})
summary["share nice"] = (summary["nice-to-have"] / summary["mentioned"]).round(2) # share of mentions that are nice-to-have
print(summary.sort_values("mentioned", ascending=False).to_string())

### 7.4 Result: most skills are required when they are mentioned

**When an ad mentions a skill, it usually requires it.** Even for the two most frequent skills, only about one mention in six is hedged: Python is nice-to-have in 83 of 487 ads (17%), SQL in 72 of 463 (16%).

| Skill | Mentioned | Required | Nice-to-have | Share nice |
|---|---|---|---|---|
| Python | 487 | 404 | 83 | 0.17 |
| SQL | 463 | 391 | 72 | 0.16 |
| Machine learning | 318 | 299 | 19 | 0.06 |
| Data modelling | 316 | 295 | 21 | 0.07 |
| Power BI | 220 | 182 | 38 | 0.17 |
| Azure | 215 | 175 | 40 | 0.19 |
| ETL / ELT | 215 | 199 | 16 | 0.07 |
| SAP | 192 | 167 | 25 | 0.13 |
| LLM / GenAI | 151 | 147 | 4 | 0.03 |
| R | 92 | 69 | 23 | 0.25 |
| Airflow | 60 | 44 | 16 | 0.27 |
| Kafka | 41 | 30 | 11 | 0.27 |

Two patterns stand out:

- **Core methods are almost never optional:** machine learning (6%), data modelling (7%), ETL / ELT (7%) and LLM / GenAI (3%). When an ad names them, they are usually the job itself.
- **Specific tools are hedged more often:** R (25%), Airflow (27%) and Kafka (27%). Ads tend to treat them as one option among several ("Python *oder* R", "Kafka *oder* Spark") or as a bonus.

Shares for rare skills (e.g. MATLAB with 4 mentions, C# with 16) are not reliable and are left out here.

### 7.5 Checking the classification

**I read 15 random mentions labelled *nice-to-have* and 15 labelled *required*, each with its whole unit and, where relevant, its heading.** The sample is stratified because hedged mentions are rare: a plain random sample would contain almost none of them. For each mention, the question was whether the label fits what the ad says about this skill.

Each mention is printed with its whole unit: the skill is marked `>>> <<<`, hedging words `[[ ]]`. The judgments are recorded in the cell after the printout.

In [ ]:
# All skill mentions at unit level: one row per (unit, skill)
rows = [(i, skill) for skill, p in SKILLS.items()
        for i in units.index[units["unit"].str.contains(p, regex=True)]]
mentions = pd.DataFrame(rows, columns=["unit_id", "skill"]).join(units, on="unit_id")

# Heading text for each section, to show why a unit counts as hedged
headings = units[units["heading"]].set_index(["ad", "section"])["unit"]

# 15 hedged and 15 unhedged mentions (stratified, because hedged mentions are rare)
sample = pd.concat([mentions[mentions["hedged"]].sample(15, random_state=1),
                    mentions[~mentions["hedged"]].sample(15, random_state=1)])

# Numbered output with the whole unit; skill marked >>> <<<, hedge words marked [[ ]]
def mark_hedges(t):
    return re.sub(HEDGE, lambda m: "[[" + m.group() + "]]", t)

for n, (_, r) in enumerate(sample.iterrows(), start=1):
    label = "NICE" if r["hedged"] else "REQUIRED"
    unit = mark_hedges(context(r["unit"], SKILLS[r["skill"]], width=400))
    print(f"{n:>2}. [{label}] {r['skill']}")
    if r["in_hedged_section"]:
        print("    heading:", headings.get((r["ad"], r["section"]), "?").strip())
    print("   ", unit)
    print()

In [ ]:
# Manual check: does the label fit what the ad says about this skill?
# True = correct, False = wrong
hedge_judgments = {
     1: False,  2: True,  3: True,  4: True,  5: True,
     6: True,   7: True,  8: True,  9: True, 10: True,
    11: True,  12: True, 13: False, 14: True, 15: True,   # 1-15: labelled NICE
    16: True,  17: True, 18: True, 19: True, 20: True,
    21: True,  22: True, 23: True, 24: True, 25: True,
    26: True,  27: True, 28: True, 29: True, 30: True,   # 16-30: labelled REQUIRED
}

# Reason for each wrong or unsure label
hedge_errors = {
     1: " dbt is to be zentrales Transformations-Werkzeug",
    13: "multiple skills in one unit where the hedge only refers to some of them, but not DWH",
}

j = pd.Series(hedge_judgments, dtype="object")
for name, part in [("NICE (1-15)", j.loc[1:15]), ("REQUIRED (16-30)", j.loc[16:30]), ("total", j)]:
    print(f"{name:17} correct: {(part == True).sum():>2}   wrong: {(part == False).sum()}   unsure: {part.isna().sum()}")
print()
print(pd.Series(hedge_errors, name="reason").to_string())

**28 of 30 labels are correct: all 15 *required* labels and 13 of 15 *nice-to-have* labels.** The two errors show the method's two weak points:

- **A hedge word that refers to another skill in the same unit:** "Du bringst fundierte Kenntnisse in Datenintegration, DWH-Strukturen und SQL mit und hast *idealerweise* Erfahrung mit BigQuery". The hedge applies to BigQuery, but the whole unit counts as hedged, so the data warehouse is wrongly *nice-to-have*.
- **A section that continues past its content:** dbt appears below "Nice-to-have:", but the unit describes dbt as the central tool of the team's stack ("zentrales Transformations-Werkzeug"), not as an optional skill.

Both errors turn *required* skills into *nice-to-have*, so the nice-to-have shares below are, if anything, slightly too high. Neither error comes from splitting the text into units, so a more advanced sentence splitter (e.g. spaCy) would not fix them; finding which skill a hedge refers to would need a **syntactic analysis** (as future work).

For the statistics mentions, the rule from the skill check applies: a field of study counts as a skill mention. "Studium, *idealerweise* in Statistik" is therefore a correct *nice-to-have*: the degree is required, the field is preferred.

## 8. Entry barriers

Three requirements decide how open an ad is to newcomers: **experience, degree and German level.** Each is searched in the ad text with a short list of patterns:

- **Experience:** the smallest number of years named together with "Erfahrung/experience", in either order ("mindestens 3 Jahre Berufserfahrung", "Berufserfahrung (mind. 5-7 Jahre)"); otherwise "mehrjährig/langjährig" without a number. Entry-friendly wording ("erste Erfahrungen", "Berufseinsteiger", "Quereinsteiger", "Absolvent") is recorded as a separate signal.
- **Degree:** any university degree (Studium, Bachelor, Master, degree), a PhD, and whether an alternative path is accepted ("oder vergleichbare Qualifikation", "or equivalent", "abgeschlossene Ausbildung").
- **German:** whether German is mentioned, and how the requirement is worded: a CEFR level if one is stated, otherwise "fließend / verhandlungssicher / muttersprachlich", "sehr gut", "gut" or "Grundkenntnisse". The categories follow the wording of the ad, because descriptions like "gute Deutschkenntnisse" cannot be mapped to CEFR levels. If an ad mentions German more than once, the strongest wording counts. A separate flag marks ads that accept the willingness to learn German instead.

### 8.1 Extraction

In [ ]:
# --- Experience ---------------------------------------------------------------
# Number before "Erfahrung": "3 Jahre Berufserfahrung", "5+ years of experience"
EXP_NUM = (r"(?i)(?<!\d)(\d{1,2})\s*\+?\s*(?:(?:-|–|bis|to)\s*\d{1,2}\s*)?(?:jahre?n?|years?)\b"
           r"[^.\n]{0,40}?(?:erfahrung|experience)")
# Number after "Erfahrung": "Berufserfahrung (mind. 5-7 Jahre)", "experience: at least 3 years"
EXP_NUM_AFTER = (r"(?i)(?:erfahrung|experience)[^.\n]{0,40}?\(?\s*(?:mind(?:estens|\.)?|min\.?|at least|ca\.?)?\s*"
                 r"(?<!\d)(\d{1,2})\s*\+?\s*(?:(?:-|–|bis|to)\s*\d{1,2}\s*)?(?:jahre?n?|years?)\b")
SEVERAL = r"(?i)mehrjährig|langjährig|several years|many years|extensive experience|umfangreiche\s+(?:berufs)?erfahrung"
ENTRY = (r"(?i)berufseinsteig|erste\w*\s+(?:berufs|praxis)?erfahrung|first (?:professional |hands-on )?experience"
         r"|young professional|entry.?level|quereinsteig|career (?:changer|switcher)|absolvent|graduate(?!\s+degree)"
         r"|keine berufserfahrung|no (?:prior )?experience")

y_before = jobs["text"].str.extractall(EXP_NUM)[0].astype(int)
y_after = jobs["text"].str.extractall(EXP_NUM_AFTER)[0].astype(int)
years = pd.concat([y_before, y_after])
years = years[years.between(1, 15)]                                  # drop implausible numbers
jobs["exp_years"] = years.groupby(level=0).min().reindex(jobs.index)  # lowest stated = entry threshold

several = jobs["text"].str.contains(SEVERAL)
jobs["exp_level"] = pd.cut(jobs["exp_years"], bins=[0, 2, 4, 99], labels=["1-2", "3-4", "5+"]).astype(str)
jobs.loc[jobs["exp_years"].isna() & several, "exp_level"] = "several years (no number)"
jobs.loc[jobs["exp_years"].isna() & ~several, "exp_level"] = "not stated"
jobs["entry_signal"] = jobs["text"].str.contains(ENTRY)

# ads that only got a usable number through the second pattern (for the re-check)
y_after_ok = y_after[y_after.between(1, 15)]
new_only = y_after_ok.index.get_level_values(0).unique().difference(y_before.index.get_level_values(0).unique())

# --- Degree -------------------------------------------------------------------
DEGREE = r"(?i)studium|hochschul|universit|bachelor|master|diplom|\bdegree\b|\bm\.?\s?sc\b|\bb\.?\s?sc\b"
PHD = r"(?i)promotion|\bphd\b|ph\.\s?d|doktor|doctorate"
ALT_PATH = (r"(?i)vergleichbare\s+(?:qualifikation|ausbildung|kenntnisse)|gleichwertige|or equivalent"
            r"|equivalent (?:practical )?experience|abgeschlossene\s+(?:berufs)?ausbildung")
jobs["degree"] = jobs["text"].str.contains(DEGREE)
jobs["phd"] = jobs["text"].str.contains(PHD)
jobs["alt_path"] = jobs["text"].str.contains(ALT_PATH)

# --- German -------------------------------------------------------------------
DE = (r"(?i)deutschkenntnis|deutsch(?:e|en|er)?\s+sprach|sprache\s+deutsch|deutschsprachig"
      r"|deutsch-\s*(?:und|&)\s*englisch|englisch-\s*(?:und|&)\s*deutsch"
      r"|deutsch\s+(?:und|sowie|&)\s+englisch|(?:englisch|english)\s+(?:und|sowie|&|and)\s+(?:deutsch|german)"
      r"|deutsch\s*\(?[abc][12]\b|german\s+(?:language|skills|proficiency|\(?[abc][12]\b)"
      r"|(?:fluent|fließend\w*)\s+(?:in\s+)?(?:german|deutsch)|(?:german|deutsch)\s*(?:-|:)?\s*(?:fließend|verhandlungssicher|muttersprach)"
      r"|(?:verhandlungssicher|muttersprachlich)\w*\s+(?:deutsch|german)")

# Level by the wording itself, not mapped to CEFR ("gute Kenntnisse" is not B2).
# Order = priority: an ad gets the strongest level found in any of its German units.
GERMAN_LEVELS = [
    ("CEFR C1/C2",      r"(?i)\bc[12]\b"),
    ("CEFR B1/B2",      r"(?i)\bb[12]\b"),
    ("fluent / native", r"(?i)fließend|verhandlungssicher|muttersprach|fluent|native"),
    ("very good",       r"(?i)sehr gut|very good|excellent|ausgezeichnet|hervorragend|\bsicher(?:e|en|er|es)?\b|proficien"),
    ("basic",           r"(?i)grundkenntnis|basic"),
    ("good",            r"(?i)\bgut(?:e|en|er|es)?\b|\bgood\b|solid"),
]
GERMAN_LABELS = [label for label, _ in GERMAN_LEVELS] + ["mentioned, no level"]

def german_level(unit):
    for i, (_, pattern) in enumerate(GERMAN_LEVELS):
        if re.search(pattern, unit):
            return i
    return len(GERMAN_LEVELS)

# Own name for these units, so the unit table from section 7 is not overwritten
text_units = jobs["text"].str.split(SPLIT, regex=True).explode()
de_units = text_units[text_units.str.contains(DE, na=False)]
de_level = de_units.map(german_level)
jobs["german"] = (de_level.groupby(level=0).min().map(dict(enumerate(GERMAN_LABELS)))
                  .reindex(jobs.index).fillna("not mentioned"))

# German required OR willingness to learn it (only inside units that mention German)
DE_LEARN = (r"(?i)bereitschaft[^.\n]{0,60}?(?:zu erlernen|zu lernen|zu verbessern)"
            r"|willing\w*\s+to\s+learn|(?:erlernen|learn)\w*[^.\n]{0,30}?(?:deutsch|german)")
learn_units = de_units[de_units.str.contains(DE_LEARN)]
jobs["german_learnable"] = jobs.index.isin(learn_units.index)

# --- Summary ------------------------------------------------------------------
print(jobs["exp_level"].value_counts(), "\n")
print(f"{jobs['entry_signal'].sum()} ads with an entry-friendly signal")
print(f"{len(new_only)} ads got their number only from the second pattern\n")
print(jobs[["degree", "phd", "alt_path"]].sum(), "\n")
print(jobs["german"].value_counts(), "\n")
print(f"{jobs['german_learnable'].sum()} ads accept willingness to learn German")

**Most ads give no number of years.** 613 of 1,062 ads state no experience requirement, and 282 ask for several years without a number. Of the 167 ads with a number, 68 ask for 5 or more years, 47 for 3–4 and 52 for 1–2; for 20 of them the number is found only by the second pattern ("Erfahrung (mind. 5-7 Jahre)"). 153 ads contain an entry-friendly signal.

**A degree is the norm, but often not a must:** 821 ads mention a university degree, 338 name an alternative path, and 68 mention a PhD.

**German is mentioned in 570 ads.** In 451 of them the strongest wording is "sehr gut" or higher (very good 215, fluent / native 126, CEFR C1/C2 110); 51 ask for "gut", 25 for B1/B2 and 42 give no level. Only 4 ads accept the willingness to learn German.

### 8.2 Checking the extraction

Random samples are read by hand and numbered:
- 5 ads per experience and degree category
- 5 ads whose number is found only by the second experience pattern
- 3 units per German category
- the units that accept willingness to learn German

The question for each category is listed in the judgment cell, together with a reason for every wrong or unsure item.

In [ ]:
# --- Manual check: random samples per category, numbered for the judgments below ---
def show(n, text, pattern, width=80, extra=""):
    s, e = re.search(pattern, text).span()
    print(f"{n:>2}.{extra} ...{text[max(0, s - width):s]}>>>{text[s:e]}<<<{text[e:e + width]}...\n")


EXP_ANY = EXP_NUM + "|" + EXP_NUM_AFTER.removeprefix("(?i)")   # both experience patterns
item_category = {}                                            # item number -> category

CHECKS = [("exp_years", EXP_ANY, jobs["exp_years"].notna()),
          ("several years", SEVERAL, jobs["exp_level"] == "several years (no number)"),
          ("entry signal", ENTRY, jobs["entry_signal"]),
          ("degree", DEGREE, jobs["degree"]),
          ("phd", PHD, jobs["phd"]),
          ("alt path", ALT_PATH, jobs["alt_path"])]

n = 0
for name, pattern, mask in CHECKS:
    print(f"=== {name} ({mask.sum()} ads)")
    for i in jobs.index[mask].to_series().sample(min(5, mask.sum()), random_state=1):
        n += 1
        item_category[n] = name
        extra = f" -> {int(jobs.at[i, 'exp_years'])} years" if name == "exp_years" else ""
        print(f"{n:>2}.{extra}", context(jobs.at[i, "text"], pattern, width=80))
    print()

print("=== exp_years: number found only by the second pattern ===\n")
for idx in pd.Series(new_only).sample(min(5, len(new_only)), random_state=1):
    n += 1
    item_category[n] = "exp_years (2nd pattern)"
    show(n, jobs.at[idx, "text"], EXP_NUM_AFTER, extra=f" -> {int(jobs.at[idx, 'exp_years'])} years")

for level, label in enumerate(GERMAN_LABELS):
    pool = de_level[de_level == level]
    print(f"=== German: {label} ({len(pool)} units) ===\n")
    for idx in pool.sample(min(3, len(pool)), random_state=1).index:
        n += 1
        item_category[n] = f"German: {label}"
        unit = de_units[de_level == level].loc[[idx]].iloc[0]
        print(f"{n:>2}. {unit.strip()[:250]}\n")

print(f"=== Willingness to learn German ({len(learn_units)} units) ===\n")
for unit in learn_units.sample(min(5, len(learn_units)), random_state=1):
    n += 1
    item_category[n] = "German: willing to learn"
    print(f"{n:>2}. {unit.strip()[:250]}\n")

In [ ]:
# Manual check of the entry barriers: True = correct, False = wrong, None = unsure
# The question per category:
#   exp_years                is the number the minimum years of professional experience the job requires?
#   several years            does the ad require several years of experience (without a number)?
#   entry signal             does the phrase signal that the ad is open to newcomers?
#   degree                   does the ad ask for a university degree?
#   phd                      does the ad ask for or welcome a PhD?
#   alt path                 does the ad accept an alternative to a degree?
#   German: <level>          does the label match how the ad words its German requirement?
#   German: willing to learn does the ad accept willingness to learn German instead of German skills?

barrier_judgments = {
    1: True,  2: True,  3: False,  4: True,  5: True,    # exp_years (new sample: judge)
    6: True,  7: True,  8: True,  9: True, 10: True,    # several years 
    11: None, 12: True, 13: True, 14: True, 15: True,    # entry signal
    16: True, 17: True, 18: True, 19: True, 20: True,    # degree
    21: True, 22: True, 23: True, 24: True, 25: True,    # phd
    26: True, 27: True, 28: True, 29: True, 30: True,    # alt path
    31: True, 32: True, 33: True, 34: True, 35: True,    # exp_years (2nd pattern)
    36: True, 37: True, 38: False,                        # German: CEFR C1/C2
    39: True, 40: True, 41: True,                        # German: CEFR B1/B2
    42: True, 43: None, 44: True,                        # German: fluent / native
    45: True, 46: True, 47: True,                        # German: very good
    48: False,                                            # German: basic
    49: True, 50: True, 51: True,                        # German: good
    52: True, 53: True, 54: True,                        # German: mentioned, no level
    55: True, 56: True, 57: True, 58: True,              # German: willing to learn
}

# Reason for each wrong or unsure item
barrier_errors = {
    3: "5 years is mentioned, but only part of the requirement 'Mindestens  >>>5 Jahre Callcenter- und 3 Jahre BI-Reporting-Erfahrung<<< erforderlich'",
    11: "the ad signals openness to newcomers, but when it comes to 'pharmazeutischen Umfeld'",
    38: " Instead of German C1/C1, the ad requires Deutsch (B2/C1)",
    43: "the add mentions German skills are a plus instead of requiring fluency/native",
    48: "The ads describes the training the applicant will receive, including on-site sponsored German language courses, presumably as a benefit, not a requirement to learn German"
}

j = pd.DataFrame({"category": pd.Series(item_category),
                  "judgment": pd.Series(barrier_judgments, dtype="object")})
result = j.groupby("category", sort=False)["judgment"].agg(
    correct=lambda s: (s == True).sum(), wrong=lambda s: (s == False).sum(), unsure=lambda s: s.isna().sum())
print(result.to_string())
print(f"\ntotal: {result['correct'].sum()} correct, {result['wrong'].sum()} wrong, "
      f"{result['unsure'].sum()} unsure of {len(j)}\n")
print(pd.Series(barrier_errors, name="reason", dtype="object").to_string())

**Result of the manual check:** 53 of 58 items correct (91%), 3 wrong, 2 unsure. Degree, PhD, alternative path and the second experience pattern were correct in every sampled case.

| Item | Category | Problem | Consequence |
|---|---|---|---|
| 3 | exp_years | "5 Jahre Callcenter- und 3 Jahre BI-Reporting-Erfahrung": the number belongs to only part of the requirement | Experience in a different field can count as the data requirement |
| 38 | German C1/C2 | "Deutsch (B2/C1)": a range is assigned its upper level, because C is checked first | Range requirements look stricter than they are |
| 43 | German fluent | "Fluent English required, German skills are a plus": the level word belongs to English | German that is only a plus counts as required |
| 48 | German basic | The ad offers German courses; "basic" refers to living expenses | An offer counts as a requirement |
| 11 | entry signal (unsure) | "Erste Erfahrungen im pharmazeutischen Umfeld": newcomer-friendly only within one domain | Entry signals can be domain-specific |

Items 43 and 48 show the same limitation as the skills: the extraction finds the right words, but cannot always tell what they refer to. Like the hedge scope for skills, this is left for future work (syntactic analysis). The German level is reported with this limitation; the other categories can be used as they are.

## 9. Checkpoint

Everything the analysis needs is saved in one table, one row per ad:
- ad-level columns: role, seniority, `exp_years`, `exp_level`, `entry_signal`, `degree`, `phd`, `alt_path`, `german`, `german_learnable`
- skills as True/False columns: `skill_` (mentioned at all), `req_` (mentioned as required), `nice_` (only nice-to-have)

In [ ]:
jobs_out = jobs.join([S.add_prefix("skill_"), REQ.add_prefix("req_"), NICE.add_prefix("nice_")])
out = PROCESSED_DIR / "jobs_extracted.pkl"
jobs_out.to_pickle(out)
print(jobs_out.shape, "->", out.name)

## 10. Summary and future work

| Step | Result | Manual check |
|---|---|---|
| Data-role ads | 1,062 (6 ads removed that matched only through the name "NTT DATA") | – |
| Role | 6 roles; Data Engineer is the most frequent (281 ads) | 4 plausibility checks, no hand labels |
| Seniority | 757 not stated, 250 senior, 55 junior | sample titles per level |
| Skills | 46 skills; Python (46%) and SQL (44%) are the most frequent | 108 of 110 hits correct |
| Required vs. nice-to-have | most mentions are required; R, Airflow and Kafka are hedged most often | 28 of 30 labels correct |
| Entry barriers | experience, degree, German level | 53 of 58 items correct, 2 unsure |

**Future work noted in this notebook:**

- Move the "NTT DATA" prefix removal into the filter in `02_cleaning`.
- Context-aware matching for Git, e.g. ignoring GitHub in application instructions.
- Syntactic analysis to find what a hedging word or a level word refers to ("Fluent English required, German skills are a plus").
- Ranges such as "Deutsch (B2/C1)": use the lower level.
- A text classifier as a comparison to the rule-based extraction.